In [1]:
import sys
sys.path.append("..")

In [4]:
from src.data.loader import get_dataloaders
from src.data.transforms import get_baseline_train_transforms, get_baseline_val_transforms, get_baseline_test_transforms

In [6]:
train_tfms = get_baseline_train_transforms(img_size=256)
val_tfms   = get_baseline_val_transforms(img_size=256)
test_tfms  = get_baseline_test_transforms(img_size=256)

train_loader, val_loader, test_loader = get_dataloaders(cfg_path="../configs/default.yaml",
                    train_tfms=train_tfms,
                    val_tfms=val_tfms,
                    test_tfms=test_tfms)

In [8]:
image, mask = next(iter(train_loader))

In [13]:
import time
time_ls = []
start_time = time.time()
t0 = time.time()
counter = 0
for image, mask in train_loader:
    print(image.shape, mask.shape)
    t1 =time.time()
    tt = t1 - t0
    time_ls.append(tt)
    if counter >= 4:
        break
    t0 = time.time()
    counter += 1
    print(f"Iteration time: {tt:.4f} seconds")

end_time = time.time()
total_time = end_time - start_time
print(f"Total time for loop: {total_time:.4f} seconds")
print(f"Average iteration time: {sum(time_ls) / len(time_ls):.4f} seconds")

torch.Size([8, 3, 256, 256]) torch.Size([8, 256, 256])
Iteration time: 1.9568 seconds
torch.Size([8, 3, 256, 256]) torch.Size([8, 256, 256])
Iteration time: 0.0159 seconds
torch.Size([8, 3, 256, 256]) torch.Size([8, 256, 256])
Iteration time: 0.0002 seconds
torch.Size([8, 3, 256, 256]) torch.Size([8, 256, 256])
Iteration time: 0.0003 seconds
torch.Size([8, 3, 256, 256]) torch.Size([8, 256, 256])
Total time for loop: 22.0010 seconds
Average iteration time: 0.3987 seconds


Total time for loop: 21.9577 seconds
Average iteration time: 0.3887 seconds


Exploring Pretrained models to see how to ingegrate them with unet model.

In [14]:
import torch
from torchvision import models

# Load ResNet34 with ImageNet pretrained weights
model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)

# Set the model to evaluation mode (optional but common for inference)
# model.eval()

# Example: move to device if needed
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# Example: dummy input for testing
# x = torch.randn(1, 3, 224, 224).to(device)
# y = model(x)
# print(y.shape)


In [ ]:

model.layer1

Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (1): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (2): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, mome

In [20]:
rand_im = torch.rand(2, 3, 256, 256)
rand_im.size()

torch.Size([2, 3, 256, 256])

In [21]:
output = model(rand_im)

In [28]:
x = rand_im
print(x.shape)
print()
for layer in model.children():
    x = layer(x)
    print(layer)
    print(" -> ", x.size())
    print()
    if x.size()[-1] == 8:
        break

torch.Size([2, 3, 256, 256])

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
 ->  torch.Size([2, 64, 128, 128])

BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
 ->  torch.Size([2, 64, 128, 128])

ReLU(inplace=True)
 ->  torch.Size([2, 64, 128, 128])

MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
 ->  torch.Size([2, 64, 64, 64])

Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (1): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05,

In [29]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [31]:
conv1 = model.conv1
bn1 = model.bn1
relu1 = model.relu
mp1 = model.maxpool

layer1 = model.layer1
layer2 = model.layer2
layer3 = model.layer3
layer4 = model.layer4

layers = [conv1, bn1, relu1, mp1,
          layer1, layer2, layer3, layer4]

In [32]:
x = rand_im
print(x.shape)
for layer in layers:
    x = layer(x)
    print(layer)
    print(" -> ", x.size())
    print()

torch.Size([2, 3, 256, 256])
Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
 ->  torch.Size([2, 64, 128, 128])

BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
 ->  torch.Size([2, 64, 128, 128])

ReLU(inplace=True)
 ->  torch.Size([2, 64, 128, 128])

MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
 ->  torch.Size([2, 64, 64, 64])

Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (1): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, 

In [ ]:
import torch


In [ ]:
import torch.nn as nn

class EncoderBlock(nn.Module):
    def __init__(self, resnet_model):
        super().__init__()
        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu1 = model.relu
        self.mp1 = model.maxpool

        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4


    def forward(self, x):
        x1 = self.relu1(self.bn1(self.conv1(x))) # x -> x1: (B, 64, 128, 128)
        x2 = self.layer1(self.mp1(x1)) # x1 -> x2: (B, 64, 64, 64)
        x3 = self.layer2(x2) # x2 -> x3: (B, 128, 32, 32)
        x4 = self.layer3(x3) # x3 -> x4: (B, 256, 16, 16)
        x5 = self.layer4(x4) # x4 -> x5: (B, 512, 8, 8)
        return x1, x2, x3, x4, x5
    
    

class UpSampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu1 = nn.ReLU(inplace=True)

    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat((x, skip), dim=1)  # Concatenate along channel dimension
        x = self.relu1(self.bn1(self.conv1(x)))
        return x
    
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = UpSampleBlock(512 + 256, 256)
        self.up3 = UpSampleBlock(256 + 128, 128)
        self.up2 = UpSampleBlock(128 + 64, 64)
        self.up1 = UpSampleBlock(64 + 64, 64)
        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)  # Assuming binary segmentation

    def forward(self, x1, x2, x3, x4, x5):
        """
        x1: (B, 64, 128, 128)
        x2: (B, 64, 64, 64)
        x3: (B, 128, 32, 32)
        x4: (B, 256, 16, 16)
        x5: (B, 512, 8, 8)
        """
        d4 = self.up4(x5, x4)  # x5 and x4
        d3 = self.up3(d4, x3)  # d4 and x3
        d2 = self.up2(d3, x2)  # d3 and x2
        d1 = self.up1(d2, x1)  # d2 and x1
        out = self.final_conv(d1)
        return out
    

class ResNet34UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderBlock(model)
        self.decoder = Decoder()

    def forward(self, x):
        x1, x2, x3, x4, x5 = self.encoder(x)
        out = self.decoder(x1, x2, x3, x4, x5)
        return out


class SegmentationWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        input_size = x.size()[2:]  # H, W
        x = self.base_model(x)
        x = nn.functional.interpolate(x, size=input_size, mode='bilinear', align_corners=True)
        return x

        

In [38]:
resunet = ResNet34UNet()

In [39]:
output = resunet(rand_im)

In [41]:
output.shape

torch.Size([2, 1, 128, 128])

In [42]:
import os
cpu_cores = os.cpu_count()
print(cpu_cores)

10
